# EDA: Heart Disease Playground

Purpose: move through the requested plan Phases 1-4 via the shortest path.
Target data: `data/raw/train.csv`, `data/raw/test.csv`, `data/raw/sample_submission.csv`


### Package import

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Prefer Kaggle input paths if available; fallback to local data/raw.
kaggle_dir = Path("/kaggle/input/playground-series-s6e2")
if kaggle_dir.exists():
    train = pd.read_csv(kaggle_dir / "train.csv")
    test = pd.read_csv(kaggle_dir / "test.csv")
    sub = pd.read_csv(kaggle_dir / "sample_submission.csv")
else:
    # Infer project root (so it works even when run from nb/)
    cwd = Path.cwd().resolve()
    project_root = None
    for p in [cwd] + list(cwd.parents):
        if (p / "data" / "raw" / "train.csv").exists():
            project_root = p
            break
    if project_root is None:
        raise FileNotFoundError("project root not found: data/raw/train.csv")

    data_dir = project_root / "data" / "raw"
    train = pd.read_csv(data_dir / "train.csv")
    test = pd.read_csv(data_dir / "test.csv")
    sub = pd.read_csv(data_dir / "sample_submission.csv")

print(train.shape, test.shape, sub.shape)


### Data download

In [ ]:
display(train.head())
display(test.head())
display(sub.head())

In [ ]:
# Shapes
print("train:", train.shape)
print("test:", test.shape)

# Column diffs
train_cols = set(train.columns)
test_cols = set(test.columns)
print("Only in train:", sorted(train_cols - test_cols))
print("Only in test:", sorted(test_cols - train_cols))

# dtypes
train.dtypes.to_frame("dtype").head(30)


### Data preprocessing

In [ ]:
id_col = "id" if "id" in train.columns else None
if id_col:
    print("train dup id:", train[id_col].duplicated().sum())
    print("test dup id:", test[id_col].duplicated().sum())
    train_ids = set(train[id_col])
    test_ids = set(test[id_col])
    print("train/test overlap:", len(train_ids & test_ids))
else:
    print("No id column found.")


In [ ]:
def missing_zero_table(df: pd.DataFrame) -> pd.DataFrame:
    missing = df.isna().mean().rename("missing_rate")
    uniq = df.nunique(dropna=True).rename("n_unique")
    dtype = df.dtypes.rename("dtype")

    num_cols_ = df.select_dtypes(include=["number"]).columns
    zero_rate = pd.Series(index=df.columns, dtype="float64", name="zero_rate")
    zero_rate.loc[num_cols_] = (df[num_cols_] == 0).mean()

    out = pd.concat([dtype, uniq, missing, zero_rate], axis=1)
    return out.sort_values(["missing_rate", "zero_rate"], ascending=False)

mz = missing_zero_table(train)
mz.head(30)


### Data labeling

In [ ]:
# Infer target column
target_col = None
for cand in ["Heart Disease", "HeartDisease", "target", "label"]:
    if cand in train.columns:
        target_col = cand
        break
print("target_col:", target_col)

if target_col:
    display(train[target_col].value_counts(dropna=False))
    display(train[target_col].value_counts(normalize=True, dropna=False))
    print("dtype:", train[target_col].dtype)


In [ ]:
TARGET_RAW = "Heart Disease"
target_map = {"Absence": 0, "Presence": 1}

y = train[TARGET_RAW].map(target_map)
assert y.isna().sum() == 0

print("pos_rate:", float(y.mean()))

ID_COL = "id"
FEATURE_COLS = [c for c in train.columns if c not in [target_col, ID_COL]]

cat_cols = [c for c in FEATURE_COLS if train[c].nunique(dropna=True) <= 20]
num_cols = [
    c for c in FEATURE_COLS if pd.api.types.is_numeric_dtype(train[c]) and c not in cat_cols
]

for c in cat_cols:
    train[c] = train[c].astype("category")
    test[c] = test[c].astype("category")


### Categorical candidates

In [ ]:
# Numeric columns with few uniques are categorical candidates
n_unique = train[FEATURE_COLS].nunique(dropna=True).sort_values()
small_unique = n_unique[n_unique <= 20]
small_unique


### Numeric summary

In [ ]:
train[num_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T


### Relationship to target

- Numeric: mean/median by target
- Categorical: target rate by category


In [ ]:
if target_col:
    # Numeric
    if len(num_cols) > 0:
        display(train.assign(y=y).groupby("y")[num_cols].agg(["mean", "median"]).T)

    def pos_rate_by_category(col: str) -> pd.DataFrame:
        tmp = pd.DataFrame({"x": train[col], "y": y})
        return (
            tmp.groupby("x")["y"]
            .agg(pos_rate="mean", count="size")
            .sort_values(["pos_rate", "count"], ascending=[False, False])
        )

    for c in cat_cols:
        display(pos_rate_by_category(c))


In [ ]:
for c in num_cols:
    if (train[c] == 0).any():
        rate0 = y[train[c] == 0].mean()
        rate1 = y[train[c] != 0].mean()
        print(
            f"{c:25s} zero_rate={(train[c]==0).mean():.3f}  pos_rate(0)={rate0:.3f}  pos_rate(!0)={rate1:.3f}"
        )


In [ ]:
zero_sensitive = ["ST depression", "Number of vessels fluro"]
for c in zero_sensitive:
    train[f"{c}__is_zero"] = (train[c] == 0).astype("int8")
    test[f"{c}__is_zero"] = (test[c] == 0).astype("int8")

# --- interaction features ---
if "Age" in train.columns and "Sex" in train.columns:
    train["Age_X_Sex"] = train["Age"].astype(str) + "_" + train["Sex"].astype(str)
    test["Age_X_Sex"] = test["Age"].astype(str) + "_" + test["Sex"].astype(str)

if "Max HR" in train.columns and "Age" in train.columns:
    train["MaxHR_div_Age"] = (train["Max HR"] / (train["Age"] + 1)).round(2)
    test["MaxHR_div_Age"] = (test["Max HR"] / (test["Age"] + 1)).round(2)
    train["MaxHR_vs_Expected"] = train["Max HR"] - (220 - train["Age"])
    test["MaxHR_vs_Expected"] = test["Max HR"] - (220 - test["Age"])

if "ST depression" in train.columns and "Slope of ST" in train.columns:
    train["STdep_X_Slope"] = train["ST depression"].round(1).astype(str) + "_" + train["Slope of ST"].astype(str)
    test["STdep_X_Slope"] = test["ST depression"].round(1).astype(str) + "_" + test["Slope of ST"].astype(str)

if "Thallium" in train.columns and "Chest pain type" in train.columns:
    train["Thallium_X_ChestPain"] = train["Thallium"].astype(str) + "_" + train["Chest pain type"].astype(str)
    test["Thallium_X_ChestPain"] = test["Thallium"].astype(str) + "_" + test["Chest pain type"].astype(str)

if "Number of vessels fluro" in train.columns and "Thallium" in train.columns:
    train["Vessels_X_Thallium"] = train["Number of vessels fluro"].astype(str) + "_" + train["Thallium"].astype(str)
    test["Vessels_X_Thallium"] = test["Number of vessels fluro"].astype(str) + "_" + test["Thallium"].astype(str)

if "EKG results" in train.columns and "Exercise angina" in train.columns:
    train["EKG_X_Angina"] = train["EKG results"].astype(str) + "_" + train["Exercise angina"].astype(str)
    test["EKG_X_Angina"] = test["EKG results"].astype(str) + "_" + test["Exercise angina"].astype(str)

# --- bucketed features ---
if "ST depression" in train.columns:
    train["STdep_bucket"] = pd.cut(
        train["ST depression"],
        bins=[-0.1, 0, 1, 2, 10],
        labels=["0", "0-1", "1-2", "2+"],
    ).astype(str)
    test["STdep_bucket"] = pd.cut(
        test["ST depression"],
        bins=[-0.1, 0, 1, 2, 10],
        labels=["0", "0-1", "1-2", "2+"],
    ).astype(str)

if "Age" in train.columns:
    train["Age_bucket"] = pd.cut(
        train["Age"],
        bins=[0, 40, 50, 60, 70, 100],
        labels=["<40", "40-50", "50-60", "60-70", "70+"],
    ).astype(str)
    test["Age_bucket"] = pd.cut(
        test["Age"],
        bins=[0, 40, 50, 60, 70, 100],
        labels=["<40", "40-50", "50-60", "60-70", "70+"],
    ).astype(str)

if "Cholesterol" in train.columns:
    train["Chol_bucket"] = pd.cut(
        train["Cholesterol"],
        bins=[0, 200, 240, 280, 1000],
        labels=["<200", "200-240", "240-280", "280+"],
    ).astype(str)
    test["Chol_bucket"] = pd.cut(
        test["Cholesterol"],
        bins=[0, 200, 240, 280, 1000],
        labels=["<200", "200-240", "240-280", "280+"],
    ).astype(str)

if "Max HR" in train.columns:
    maxhr_bins = np.unique(np.quantile(train["Max HR"].values, np.linspace(0, 1, 6)))
    if len(maxhr_bins) < 2:
        min_v = float(train["Max HR"].min())
        max_v = float(train["Max HR"].max())
        maxhr_bins = np.array([min_v - 1e-6, max_v + 1e-6])
    maxhr_bins[0] = maxhr_bins[0] - 1e-6
    maxhr_bins[-1] = maxhr_bins[-1] + 1e-6
    train["MaxHR_bucket"] = pd.cut(
        train["Max HR"],
        bins=maxhr_bins,
        labels=False,
        include_lowest=True,
    ).astype(str)
    test["MaxHR_bucket"] = pd.cut(
        test["Max HR"],
        bins=maxhr_bins,
        labels=False,
        include_lowest=True,
    ).astype(str)
    train["MaxHR_bucket"] = "HR_Q" + train["MaxHR_bucket"]
    test["MaxHR_bucket"] = "HR_Q" + test["MaxHR_bucket"]

if "BP" in train.columns:
    train["BP_bucket"] = pd.cut(
        train["BP"],
        bins=[0, 120, 140, 160, 300],
        labels=["<120", "120-140", "140-160", "160+"],
    ).astype(str)
    test["BP_bucket"] = pd.cut(
        test["BP"],
        bins=[0, 120, 140, 160, 300],
        labels=["<120", "120-140", "140-160", "160+"],
    ).astype(str)


In [ ]:
# Numeric: target-wise hist (sampled for speed)
cols = ["Age", "Max HR", "ST depression"]
tmp = train.sample(50_000, random_state=42).assign(y=y)

for c in cols:
    plt.figure()
    tmp[tmp["y"] == 0][c].hist(alpha=0.5, bins=30)
    tmp[tmp["y"] == 1][c].hist(alpha=0.5, bins=30)
    plt.title(c)
    plt.show()


In [ ]:
# Categorical: pos_rate by category
cat_plot_cols = ["Chest pain type", "Thallium", "Number of vessels fluro"]

for c in cat_plot_cols:
    if c not in train.columns:
        continue
    df = pos_rate_by_category(c)
    plt.figure(figsize=(6, 3))
    df["pos_rate"].plot(kind="bar")
    plt.title(c)
    plt.ylabel("pos_rate")
    plt.tight_layout()
    plt.show()


### Train vs Test distribution drift

- Adversarial validation (CatBoost, train-vs-test AUC)
- Categorical: rate differences


In [ ]:
from catboost import CatBoostClassifier, CatBoostError
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

use_gpu = True

adv_cols = [c for c in FEATURE_COLS if c in test.columns]
adv_cat_cols = [c for c in adv_cols if train[c].nunique(dropna=True) <= 20]

X_adv = pd.concat([
    train[adv_cols].assign(_is_test=0),
    test[adv_cols].assign(_is_test=1),
], axis=0).reset_index(drop=True)
y_adv = X_adv.pop("_is_test")

X_adv_train, X_adv_valid, y_adv_train, y_adv_valid = train_test_split(
    X_adv,
    y_adv,
    test_size=0.2,
    random_state=42,
    stratify=y_adv,
)

adv_cat_idx = [X_adv_train.columns.get_loc(c) for c in adv_cat_cols if c in X_adv_train.columns]

adv_params = dict(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=3000,
    # iterations=10,
    od_type="Iter",
    od_wait=150,
    depth=5,
    learning_rate=0.08,
    random_seed=42,
    verbose=200,
)
if use_gpu:
    adv_params.update(task_type="GPU", devices="0")
else:
    adv_params.update(task_type="CPU")

adv_model = CatBoostClassifier(**adv_params)

try:
    adv_model.fit(
        X_adv_train,
        y_adv_train,
        cat_features=adv_cat_idx,
        eval_set=(X_adv_valid, y_adv_valid),
        use_best_model=True,
    )
except CatBoostError:
    # GPU can be unavailable on some environments; keep a CPU fallback.
    cpu_params = {k: v for k, v in adv_params.items() if k not in ["task_type", "devices"]}
    cpu_params["task_type"] = "CPU"
    adv_model = CatBoostClassifier(**cpu_params)
    adv_model.fit(
        X_adv_train,
        y_adv_train,
        cat_features=adv_cat_idx,
        eval_set=(X_adv_valid, y_adv_valid),
        use_best_model=True,
    )

adv_valid_pred = adv_model.predict_proba(X_adv_valid)[:, 1]
adv_auc = roc_auc_score(y_adv_valid, adv_valid_pred)
print(f"Adversarial validation AUC (train vs test): {adv_auc:.4f}")

imp = adv_model.get_feature_importance()
drift_num = (
    pd.DataFrame(
        {
            "col": adv_model.feature_names_,
            "adversarial_importance": imp,
        }
    )
    .sort_values("adversarial_importance", ascending=False)
    .reset_index(drop=True)
)
drift_num.head(20)


### Categorical rate differences

In [ ]:
cat_cols = [c for c in FEATURE_COLS if c in test.columns and train[c].nunique(dropna=True) <= 20]

rows = []
for c in cat_cols:
    train_rate = train[c].value_counts(normalize=True)
    test_rate = test[c].value_counts(normalize=True)
    diff = (train_rate - test_rate).abs().fillna(0).sum() / 2
    rows.append((c, diff))

drift_cat = pd.DataFrame(rows, columns=["col", "total_variation_distance"]).sort_values(
    "total_variation_distance", ascending=False
)
drift_cat.head(20)


In [ ]:
from catboost import CatBoostClassifier, CatBoostError
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import numpy as np
import pandas as pd

use_gpu = bool(globals().get("use_gpu", True))

# --- categorical columns: fix them explicitly ---
cat_cols_model = [
    "Sex",
    "FBS over 120",
    "Exercise angina",
    "EKG results",
    "Slope of ST",
    "Thallium",
    "Number of vessels fluro",
    "Chest pain type",
    "Age_X_Sex",
    "STdep_X_Slope",
    "Thallium_X_ChestPain",
    "Vessels_X_Thallium",
    "EKG_X_Angina",
    "STdep_bucket",
    "Age_bucket",
    "Chol_bucket",
    "MaxHR_bucket",
    "BP_bucket",
]

feature_cols = [c for c in train.columns if c not in [target_col, ID_COL]]

X_train_full = train[feature_cols].copy()
X_test = test[feature_cols].copy()

# make categorical columns string (avoid pandas category dtype issues)
for c in cat_cols_model:
    X_train_full[c] = X_train_full[c].astype(str)
    X_test[c] = X_test[c].astype(str)

cat_idx = [X_train_full.columns.get_loc(c) for c in cat_cols_model]

X_model = X_train_full
X_test_model = X_test

def run_catboost_cv(X, y, train_params, n_splits=5, cat_features=None, tag="cv"):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    rows = []

    cat_idx = None
    if cat_features:
        cat_idx = [X.columns.get_loc(c) for c in cat_features if c in X.columns]

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), start=1):
        print(f"===== {tag} fold {fold}/{n_splits} =====")
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        fold_model = CatBoostClassifier(**train_params)
        try:
            fold_model.fit(
                X_tr,
                y_tr,
                cat_features=cat_idx,
                eval_set=(X_va, y_va),
                use_best_model=True,
            )
        except CatBoostError:
            print("[WARN] GPU failed -> fallback to CPU")
            cpu_params = {k: v for k, v in train_params.items() if k not in ["task_type", "devices"]}
            cpu_params["task_type"] = "CPU"
            fold_model = CatBoostClassifier(**cpu_params)
            fold_model.fit(
                X_tr,
                y_tr,
                cat_features=cat_idx,
                eval_set=(X_va, y_va),
                use_best_model=True,
            )

        va_pred = fold_model.predict_proba(X_va)[:, 1]
        auc = roc_auc_score(y_va, va_pred)
        best_it = fold_model.get_best_iteration()
        if best_it is None or best_it < 1:
            best_it = train_params.get("iterations", 1000)
        print(f"[{tag} fold {fold}] AUC={auc:.6f}  best_iteration={best_it}")

        rows.append({"fold": fold, "roc_auc": auc, "best_iteration": int(best_it)})

    cv_table = pd.DataFrame(rows)
    best_it_median = int(np.median(cv_table["best_iteration"]))
    final_iterations = max(1, int(best_it_median * 1.15))

    cv_stats = {
        "mean_roc_auc": float(cv_table["roc_auc"].mean()),
        "std_roc_auc": float(cv_table["roc_auc"].std(ddof=1)),
        "mean_best_iteration": float(cv_table["best_iteration"].mean()),
        "best_it_median": best_it_median,
        "final_iterations": final_iterations,
    }
    return cv_table, cv_stats

train_params = dict(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=3000,
    # iterations=30,
    od_type="Iter",
    od_wait=150,
    depth=6,
    learning_rate=0.08,
    l2_leaf_reg=3,
    random_seed=42,
    verbose=100,
)
if use_gpu:
    train_params.update(task_type="GPU", devices="0")
else:
    train_params.update(task_type="CPU")

cv_table, cv_stats = run_catboost_cv(
    X_model,
    y,
    train_params,
    n_splits=5,
    cat_features=cat_cols_model,
    tag="base",
)

best_it_median = cv_stats["best_it_median"]
final_iterations = cv_stats["final_iterations"]

print(
    f"base CV mean AUC={cv_stats['mean_roc_auc']:.6f} +/- {cv_stats['std_roc_auc']:.6f}, "
    f"best_it_median={best_it_median}, final_iterations={final_iterations}"
)
cv_table


In [ ]:
summary = pd.DataFrame(
    {
        "metric": [
            "mean_roc_auc",
            "std_roc_auc",
            "mean_best_iteration",
            "best_it_median",
            "final_iterations",
        ],
        "value": [
            cv_stats["mean_roc_auc"],
            cv_stats["std_roc_auc"],
            cv_stats["mean_best_iteration"],
            cv_stats["best_it_median"],
            cv_stats["final_iterations"],
        ],
    }
)
summary


### Fast tuning setup (3-fold + shorter early stopping)

In [ ]:
train_params_fast = dict(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=1500,
    # iterations=10,
    od_type="Iter",
    od_wait=50,
    depth=5,
    learning_rate=0.12,
    rsm=0.8,
    bootstrap_type="Bernoulli",
    subsample=0.85,
    random_seed=42,
    task_type="CPU",
    verbose=200,
)

fast_cv_table, fast_cv_stats = run_catboost_cv(
    X_model,
    y,
    train_params_fast,
    n_splits=3,
    cat_features=cat_cols_model,
    tag="fast",
)

pd.DataFrame(
    {
        "metric": [
            "mean_roc_auc",
            "std_roc_auc",
            "best_it_median",
            "final_iterations",
        ],
        "value": [
            fast_cv_stats["mean_roc_auc"],
            fast_cv_stats["std_roc_auc"],
            fast_cv_stats["best_it_median"],
            fast_cv_stats["final_iterations"],
        ],
    }
)


### Priority B/C lightweight parameter sweeps

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import pandas as pd

# Option A: fast holdout sweep (single split)
X_sw_tr, X_sw_va, y_sw_tr, y_sw_va = train_test_split(
    X_model,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

cat_idx_sw = [X_model.columns.get_loc(c) for c in cat_cols_model if c in X_model.columns]

def run_catboost_holdout(train_params, tag="holdout"):
    model = CatBoostClassifier(**train_params)
    try:
        model.fit(
            X_sw_tr,
            y_sw_tr,
            cat_features=cat_idx_sw,
            eval_set=(X_sw_va, y_sw_va),
            use_best_model=True,
        )
    except CatBoostError:
        cpu_params = {k: v for k, v in train_params.items() if k not in ["task_type", "devices"]}
        cpu_params["task_type"] = "CPU"
        model = CatBoostClassifier(**cpu_params)
        model.fit(
            X_sw_tr,
            y_sw_tr,
            cat_features=cat_idx_sw,
            eval_set=(X_sw_va, y_sw_va),
            use_best_model=True,
        )

    va_pred = model.predict_proba(X_sw_va)[:, 1]
    auc = roc_auc_score(y_sw_va, va_pred)
    best_it = model.get_best_iteration()
    if best_it is None or best_it < 1:
        best_it = train_params.get("iterations", 1000)

    return {
        "mean_roc_auc": float(auc),
        "std_roc_auc": 0.0,
        "best_it_median": int(best_it),
        "final_iterations": max(1, int(best_it * 1.15)),
        "tag": tag,
    }

# Priority B: depth x l2_leaf_reg (fast holdout)
search_rows = []
for depth in [4, 5, 6]:
    for l2_leaf_reg in [1, 3, 5, 10]:
        params = train_params_fast.copy()
        params.update(depth=depth, l2_leaf_reg=l2_leaf_reg)
        stats = run_catboost_holdout(params, tag=f"d{depth}_l2{l2_leaf_reg}")
        search_rows.append(
            {
                "phase": "holdout",
                "depth": depth,
                "l2_leaf_reg": l2_leaf_reg,
                "mean_roc_auc": stats["mean_roc_auc"],
                "std_roc_auc": stats["std_roc_auc"],
                "best_it_median": stats["best_it_median"],
                "final_iterations": stats["final_iterations"],
            }
        )

# Priority C: random_strength / bagging_temperature (fast holdout)
for random_strength in [0.0, 1.0, 2.0]:
    for bagging_temperature in [0.0, 0.5, 1.0]:
        params = train_params_fast.copy()
        params.update(
            bootstrap_type="Bayesian",
            random_strength=random_strength,
            bagging_temperature=bagging_temperature,
        )
        params.pop("subsample", None)
        stats = run_catboost_holdout(params, tag=f"rs{random_strength}_bt{bagging_temperature}")
        search_rows.append(
            {
                "phase": "holdout",
                "random_strength": random_strength,
                "bagging_temperature": bagging_temperature,
                "mean_roc_auc": stats["mean_roc_auc"],
                "std_roc_auc": stats["std_roc_auc"],
                "best_it_median": stats["best_it_median"],
                "final_iterations": stats["final_iterations"],
            }
        )

sweep_holdout = pd.DataFrame(search_rows).sort_values("mean_roc_auc", ascending=False).reset_index(drop=True)
sweep_holdout.head(12)

# Option B: re-check top candidates with lighter 3-fold CV
# top_checks = []
# for _, row in sweep_holdout.head(3).iterrows():
#     params = train_params_fast.copy()
#     params.update(iterations=800, od_wait=30)

#     if "depth" in row and pd.notna(row.get("depth")):
#         params["depth"] = int(row["depth"])
#     if "l2_leaf_reg" in row and pd.notna(row.get("l2_leaf_reg")):
#         params["l2_leaf_reg"] = float(row["l2_leaf_reg"])

#     if "random_strength" in row and pd.notna(row.get("random_strength")):
#         params.update(
#             bootstrap_type="Bayesian",
#             random_strength=float(row["random_strength"]),
#             bagging_temperature=float(row["bagging_temperature"]),
#         )
#         params.pop("subsample", None)

#     _, stats = run_catboost_cv(X_model, y, params, n_splits=3, cat_features=cat_cols_model, tag="top_recheck")
#     top_checks.append({
#         "mean_roc_auc": stats["mean_roc_auc"],
#         "std_roc_auc": stats["std_roc_auc"],
#         "best_it_median": stats["best_it_median"],
#         "final_iterations": stats["final_iterations"],
#         "params": params,
#     })

# sweep_recheck = pd.DataFrame(top_checks).sort_values(["mean_roc_auc", "std_roc_auc"], ascending=[False, True])
# sweep_recheck


### Submit prediction

In [ ]:
from catboost import CatBoostClassifier, CatBoostError
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

use_gpu = bool(globals().get("use_gpu", True))

TARGET = target_col
ID_COL = "id"

# --- categorical columns: fix them explicitly ---
cat_cols_model = [
    "Sex",
    "FBS over 120",
    "Exercise angina",
    "EKG results",
    "Slope of ST",
    "Thallium",
    "Number of vessels fluro",
    "Chest pain type",
    "Age_X_Sex",
    "STdep_X_Slope",
    "Thallium_X_ChestPain",
    "Vessels_X_Thallium",
    "EKG_X_Angina",
    "STdep_bucket",
    "Age_bucket",
    "Chol_bucket",
    "MaxHR_bucket",
    "BP_bucket",
]

feature_cols = [c for c in train.columns if c not in [TARGET, ID_COL]]

X_train_full = train[feature_cols].copy()
X_test = test[feature_cols].copy()

# make categorical columns string (avoid pandas category dtype issues)
for c in cat_cols_model:
    X_train_full[c] = X_train_full[c].astype(str)
    X_test[c] = X_test[c].astype(str)

cat_idx = [X_train_full.columns.get_loc(c) for c in cat_cols_model]

if "final_iterations" not in globals():
    if "cv_table" in globals() and not cv_table.empty:
        best_it_median = int(np.median(cv_table["best_iteration"]))
        final_iterations = max(1, int(best_it_median * 1.15))
    else:
        best_it_median = 1200
        final_iterations = int(best_it_median * 1.15)

final_params = dict(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=int(final_iterations),
    depth=6,
    learning_rate=0.08,
    l2_leaf_reg=3,
    verbose=False,
)
if use_gpu:
    final_params.update(task_type="GPU", devices="0")
else:
    final_params.update(task_type="CPU")

SEEDS = [0, 1, 2]

test_pred_all = []
oof_pred_all = []
fold_aucs_all = []

for seed in SEEDS:
    params_seed = final_params.copy()
    params_seed["random_seed"] = seed

    cv_submit = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    test_pred = np.zeros(len(X_test))
    oof_pred = np.zeros(len(X_train_full))
    fold_aucs = []

    for fold, (tr_idx, va_idx) in enumerate(cv_submit.split(X_train_full, y), start=1):
        X_tr, X_va = X_train_full.iloc[tr_idx], X_train_full.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        fold_model = CatBoostClassifier(**params_seed)
        try:
            fold_model.fit(X_tr, y_tr, cat_features=cat_idx, verbose=False)
        except CatBoostError:
            cpu_params = {k: v for k, v in params_seed.items() if k not in ["task_type", "devices"]}
            cpu_params["task_type"] = "CPU"
            fold_model = CatBoostClassifier(**cpu_params)
            fold_model.fit(X_tr, y_tr, cat_features=cat_idx, verbose=False)

        va_pred = fold_model.predict_proba(X_va)[:, 1]
        oof_pred[va_idx] = va_pred
        fold_auc = roc_auc_score(y_va, va_pred)
        fold_aucs.append(fold_auc)

        test_pred += fold_model.predict_proba(X_test)[:, 1] / cv_submit.n_splits
        print(f"[seed {seed} fold {fold}] AUC={fold_auc:.6f}")

    test_pred_all.append(test_pred)
    oof_pred_all.append(oof_pred)
    fold_aucs_all.append(fold_aucs)

# Average across seeds
if len(test_pred_all) > 1:
    test_pred = np.mean(test_pred_all, axis=0)
    oof_pred = np.mean(oof_pred_all, axis=0)
else:
    test_pred = test_pred_all[0]
    oof_pred = oof_pred_all[0]

submit = sub.copy()
submit["Heart Disease"] = test_pred

if kaggle_dir.exists():
    submit.to_csv("submission.csv", index=False)
else:
    submit.to_csv("../data/submissions/catboost_baseline.csv", index=False)

print(f"Fixed iterations from CV best_iteration median: {final_iterations} (median={best_it_median})")
print(f"OOF AUC (avg seeds): {roc_auc_score(y, oof_pred):.6f}")
submit.head()
